In [ ]:
import warnings
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from pint.errors import UnitStrippedWarning

import prism
from imagematerials.read_mym import read_mym_df

from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    GenericMaterials,
    GenericStocks,
    MaterialIntensities,
    SharesInflowStocks
)
from imagematerials.vehicles.battery import ElectricVehicleBatteries, BatteryMaterials, EvBatteryLinkModule, ElectricVehicleBatteriesWeights
from imagematerials.preprocessing import get_preprocessing_data
from imagematerials.electricity.preprocessing import get_preprocessing_data_evbattery, get_preprocessing_data_evbattery_old

from imagematerials.electricity.constants import STANDARD_SCEN_EXTERNAL_DATA

path_current = Path().resolve()
path_base = path_current.parent #.parent # base path of the project -> image-materials
path_data = Path(path_base, "data", "raw")

# Old

In [ ]:
scenario_list = {"SSP2_M_CP":("SSP2_M_CP", None),
                # "SSP2_narrow":("SSP2_narrow", "narrow"),
                 }

# scenario_base_path = Path(path_base) / 'circular_economy_scenarios'

time_start = 1971
time_end = 2100
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

all_output_old = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path(path_data, "image", climate_scen)
    circular_economy_scenario_dir = None #Path(path_base, "circular_economy_scenarios", circular_scen)

    vhc_sector = get_preprocessing_data("vehicles", path_data, 
                                    climate_policy_scenario_dir = climate_policy_scenario_dir, 
                                    circular_economy_scenario_dirs = None)
    

    factory = ModelFactory(
        [vhc_sector], complete_timeline
        ).add(GenericStocks, "vehicles"
        ).add(BatteryMaterials, "vehicles"
        ).add(GenericMaterials, "vehicles"
        )
    model_old = factory.finish()

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UnitStrippedWarning)
        model_old.simulate(simulation_timeline)
    all_output_old[scen_id] = model_old
    print(f"Finished {scen_id}")

# list(model.vehicles)

# New

In [ ]:
# Only batteries #

scenario_list = {"SSP2_M_CP":("SSP2_M_CP", None),
                # "SSP2_narrow":("SSP2_narrow", "narrow"),
                 }

# scenario_base_path = Path(path_base) / 'circular_economy_scenarios'

time_start = 1971
time_end = 2100
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

all_output_2 = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path(path_data, "image", climate_scen)
    # climate_policy_scenario_dir = Path(path_base, "IMAGE_CircoMod", climate_scen)
    circular_economy_scenario_dir = None #Path(path_base, "circular_economy_scenarios", circular_scen)
    scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA

    vhc_sector = get_preprocessing_data("vehicles", path_data, 
                                    climate_policy_scenario_dir = climate_policy_scenario_dir, 
                                    circular_economy_scenario_dirs = None)
    ev_battery_sector = get_preprocessing_data("ev_battery", path_data,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dir, cache = False, standard_scenario = scenario)     
    # elc_sector = get_preprocessing_data("electricity", path_data,
    #                                     climate_policy_scenario_dir, 
    #                                     circular_economy_scenario_dir, cache = False, standard_scenario = scenario) 
    # elc_sector is a list of preprocessing data for each electricity subsector
    

    factory = ModelFactory(
        [vhc_sector, ev_battery_sector], complete_timeline
        ).add(GenericStocks, ["vehicles"]
        ).add(ElectricVehicleBatteries, ["ev_battery"], input_sources={
        "stock_by_cohort": "vehicles",
        "inflow": "vehicles",
        "outflow_by_cohort": "vehicles"}
        ).add(GenericMaterials, "vehicles"
        )
    model_2 = factory.finish()

    model_2.simulate(simulation_timeline)
    all_output_2[scen_id] = model_2
    print(f"Finished {scen_id}")

list(model_2.ev_battery)

In [ ]:
# All 3 sectors #

scenario_list = {
                "SSP2_VLHO":("SSP2_VLHO", None),
                "SSP2_M_CP":("SSP2_M_CP", None),
                 }

# scenario_base_path = Path(path_base) / 'circular_economy_scenarios'

time_start = 1971
time_end = 2100
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path(path_data, "image", climate_scen)
    # climate_policy_scenario_dir = Path(path_base, "IMAGE_CircoMod", climate_scen)
    circular_economy_scenario_dir = None #Path(path_base, "circular_economy_scenarios", circular_scen)
    scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA

    vhc_sector = get_preprocessing_data("vehicles", path_data, 
                                    climate_policy_scenario_dir = Path(climate_policy_scenario_dir), 
                                    circular_economy_scenario_dirs = None)
    ev_battery_sector = get_preprocessing_data("ev_battery", path_data,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dir, cache = False, standard_scenario = scenario)     
    elc_sector = get_preprocessing_data("electricity", path_data,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dir, cache = False, standard_scenario = scenario) 
    # elc_sector is a list of preprocessing data for each electricity subsector
    

    factory = ModelFactory(
        [vhc_sector, ev_battery_sector, *elc_sector], complete_timeline
        ).add(GenericStocks, ["vehicles",
                              "elc_gen",
                              "elc_grid_lines",
                              "elc_grid_add",
                              "elc_stor_phs"]
        ).add(GenericMaterials, ["vehicles"]
        ).add(ElectricVehicleBatteriesWeights, ["ev_battery"], input_sources={
        "stock_by_cohort": "vehicles",
        "inflow": "vehicles",
        "outflow_by_cohort": "vehicles"}
        ).add(EvBatteryLinkModule, ["elc_stor_other"], input_sources={
        "stock_battery_kWh_v2g": "ev_battery"}
        ).add(SharesInflowStocks, ["elc_stor_other"],
        ).add(MaterialIntensities, ["elc_gen",
                                    "elc_grid_lines",
                                    "elc_grid_add",
                                    "elc_stor_phs",
                                    "elc_stor_other"
                                    ]
        )
    model = factory.finish()

    model.simulate(simulation_timeline)
    all_output[scen_id] = model
    print(f"Finished {scen_id}")
    del factory

list(model.ev_battery)

# Calculations

In [ ]:
model = all_output["SSP2_M_CP"]

# change in vehicles inflow
m1 = model.vehicles["inflow"].to_array().sum(["Region"])
m1_ev = m1.sel(Type=m1.Type.str.contains("BEV|PHEV")).sum("Type")
r_ev = m1_ev.sel(time=2050)/m1_ev.sel(time=2030)

# change in battery inflow
m2 = model.ev_battery["inflow_battery_kWh"].to_array().sum(["Region", "BatteryType", "Type"])
r_bat = m2.sel(time=2050)/m2.sel(time=2030)

# change in energy density
m3 = model.ev_battery["energy_density"]
m3_sum = m3.sum("BatteryType")
r_energy_density = m3_sum.sel(Cohort=2050)/m3_sum.sel(Cohort=2030)


In [ ]:
print("Increase from 2030 to 2050:")
print("EV vehicles inflow", round(prism.M_(r_ev).values.item(), 1))
print("Battery inflow", round(prism.M_(r_bat).values.item(), 1))
print("energy_density", prism.M_(r_energy_density).values)
print("energy density NMC in 2050", prism.M_(m3.sel(BatteryType="NMC").sel(Cohort=2050)).values)
print("energy density Lithium Ceramic in 2050", prism.M_(m3.sel(BatteryType="Lithium Ceramic").sel(Cohort=2050)).values)
print("unit energy density", prism.U_(m3))

# Plots

### external data

In [ ]:
# read the external files
df_ev_battery_i =       pd.read_csv(Path(path_data,"electricity","lit_ev_battery_inflow.csv"), index_col = [0])
df_ev_battery_s =       pd.read_csv(Path(path_data,"electricity","lit_ev_battery_stock.csv"), index_col = [0])
df_ev_vehicles_i =      pd.read_csv(Path(path_data,"electricity","lit_ev_vehicles_inflow.csv"), index_col = [0])
df_ev_vehicles_s =      pd.read_csv(Path(path_data,"electricity","lit_ev_vehicles_stock.csv"), index_col = [0])
df_storage_power_i =    pd.read_csv(Path(path_data,"electricity","lit_grid_storage_inflow_power_GW.csv"), index_col = [0])
df_storage_power_s =    pd.read_csv(Path(path_data,"electricity","lit_grid_storage_stock_power_GW.csv"), index_col = [0])
df_storage_energy_i =   pd.read_csv(Path(path_data,"electricity","lit_grid_storage_inflow_energy_GWh.csv"), index_col = [0])
df_storage_energy_s =   pd.read_csv(Path(path_data,"electricity","lit_grid_storage_stock_energy_GWh.csv"), index_col = [0])

df_ev_battery_mat_i =   pd.read_csv(Path(path_data,"electricity","lit_ev_battery_materials_inflow.csv"), index_col = [0])

# Assumptions to convert GW to GWh
duration_battery = 4 # assume 4 h duration
duration_phd = 8 # 8 h

# TIMER
path_image = Path(path_data, "image", "SSP2_M_CP", "EnergyServices")
df_energy_timer_data = read_mym_df(Path(path_data, "image", "SSP2_M_CP", "EnergyServices", "StorResTot.out"))
df_energy_timer_data_vlho = read_mym_df(Path(path_data, "image", "SSP2_VLHO", "EnergyServices", "StorResTot.out"))
df_power_timer_data = read_mym_df(Path(path_data, "image", "SSP2_M_CP", "EnergyServices", "StorCapTot.out"))
df_power_timer_data_vlho = read_mym_df(Path(path_data, "image", "SSP2_VLHO", "EnergyServices", "StorCapTot.out"))
df_energy_timer_data_seb = read_mym_df(Path(path_data, "image", "SSP2_BL", "StorResTot.out"))
df_power_timer_data_seb = read_mym_df(Path(path_data, "image", "SSP2_BL",  "StorCapTot.out"))

df_energy_timer = df_energy_timer_data.rename(columns={28: "global"})/1000 # MWh->GWh
df_energy_timer_vlho = df_energy_timer_data_vlho.rename(columns={28: "global"})/1000 # MWh->GWh
df_energy_timer_seb = df_energy_timer_data_seb.rename(columns={28: "global"})/1000 # MWh->GWh
df_power_timer = df_power_timer_data.rename(columns={28: "global"})/1000 # MW->GW
df_power_timer_seb = df_power_timer_data_seb.rename(columns={28: "global"})/1000 # MW->GW
df_power_timer_vlho = df_power_timer_data_vlho.rename(columns={28: "global"})/1000 # MW->GW

In [ ]:
df_power_timer

### Battery

#### i: battery (GWh)

In [ ]:
# INFLOW EV BATTERIES GWh #

df = df_ev_battery_i.loc[(df_ev_battery_i["Technology"] == "all_modes")&(df_ev_battery_i["unit"] == "GWh")]

# Plot ev_battery: materials over time
var = "inflow_battery_kWh"
model = all_output["SSP2_M_CP"]
m2 = model.ev_battery[var].to_array().sum(["BatteryType", "Region", "Type"]).sel(time=slice(1972,None))/1000000
model = all_output["SSP2_VLHO"]
m1 = model.ev_battery[var].to_array().sum(["BatteryType", "Region", "Type"]).sel(time=slice(1972,None))/1000000

fig, ax = plt.subplots(figsize=(8,6))
plt.plot(m1.time, m1, label="IMAGE-Materials - SSP2_VLHO", linewidth=1, c="black", linestyle = "--")
plt.plot(m2.time, m2, label="IMAGE-Materials - SSP2_M_CP", linewidth=1, c="black")
for (source, scenario), g in df.groupby(["source", "scenario"]):
    plt.scatter(g.index, g["value"], label=f"{source} – {scenario}", linewidth=1)

plt.xlim(2010,2060)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.xlabel("Year", fontsize=14)
plt.ylabel(f"Inflow EV batteries (GWh)", fontsize=14)
plt.legend()
plt.title(f"Inflow EV batteries", fontsize=14)

#### s: battery (kg)

In [ ]:
# STOCK EV BATTERIES kg #

df = df_ev_battery_s.loc[(df_ev_battery_s["Technology"] == "all_modes")&(df_ev_battery_s["unit"] == "kg")]

# Plot ev_battery: materials over time
model = all_output["SSP2_M_CP"]
m4 = model.vehicles["stock_by_cohort_materials"].sum(["material", "Region", "Type"]).sel(Time=slice(1972,None))
m3 = model.elc_stor_other["stock_by_cohort_materials"].sum(["material", "Region", "Type"]).sel(Time=slice(1972,None))
var = "stock_battery_kg"
m2 = model.ev_battery[var].to_array().sum(["BatteryType", "Region", "Type"]).sel(time=slice(1972,None)) #, "Cohort"
m21 = model.ev_battery["stock_battery_materials"].to_array().sum(["BatteryType", "Region", "Type", "material"]).sel(time=slice(1972,None))
# model = all_output["SSP2_VLHO"]
# m1 = model.ev_battery[var].to_array().sum(["BatteryType", "Region", "Type"]).sel(time=slice(1972,None)) #, "Cohort"

fig, ax = plt.subplots(figsize=(8,6))
plt.plot(m4.Time, m4, label="IMAGE-Materials - vehicles", linewidth=1, c="purple", linestyle = ":")
plt.plot(m3.Time, m3, label="IMAGE-Materials - dedicated storage", linewidth=1, c="black", linestyle = ":")
# plt.plot(m1.time, m1, label="IMAGE-Materials - SSP2_VLHO", linewidth=1, c="black", linestyle = "--")
plt.plot(m2.time, m2, label="IMAGE-Materials - EV bat.", linewidth=1, c="black") # - SSP2_M_CP
plt.plot(m21.time, m21, label="IMAGE-Materials - EV bat. - sum materials", linewidth=1, c="orange")
for (source, scenario), g in df.groupby(["source", "scenario"]):
    plt.scatter(g.index, g["value"], label=f"{source} – {scenario} - EV bat.", linewidth=1)

# plt.xlim(2010,2060)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.xlabel("Year", fontsize=14)
plt.ylabel(f"Stock EV batteries (kg)", fontsize=14)
plt.legend()
plt.title(f"Stock EV batteries (SSP2_M_CP)", fontsize=14)

#### s: battery per battery type (normalized stacked plot)

In [ ]:
# Battery stocks per battery type (normalized stacked plot)

var = "stock_battery_kWh"
m2 = (
    model.ev_battery[var]
    .to_array()
    .sum(["Region", "Type", "Cohort"])
    .sel(time=slice(1972, None))
)

# normalize to shares
m2_share = m2 / m2.sum(dim="BatteryType")

fig, ax = plt.subplots(figsize=(8, 6))

battery_types = m2_share.BatteryType.values
ax.stackplot(
    m2_share.time,
    [m2_share.sel(BatteryType=bt) for bt in battery_types],
    labels=battery_types
)

ax.set_xlabel("Year")
ax.set_ylabel("Share of total battery stock")
ax.set_ylim(0, 1)
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
ax.set_title(f"Normalized battery stock by battery type (based on stock in Wh; SSP2_M_CP)")

plt.show()


#### s: battery per battery type

In [ ]:
# Battery stocks per battery type #

var = "stock_battery_kWh"
m2 = model.ev_battery[var].to_array().sum(["Region", "Type", "Cohort"]).sel(time=slice(1972,None))

fig,ax = plt.subplots(figsize=(8,6))
for i, mat in enumerate(model.ev_battery["stock_battery_kWh"].to_array().BatteryType.values):
    plt.plot(m2.time, m2.sel(BatteryType=mat), label=str(mat), linestyle="-", linewidth=2) #"m2 - " + 
plt.xlabel("Year")
plt.ylabel(f"{var} in EV batteries (kWh)")
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
plt.title(f"{var} in EV batteries (SSP2_M_CP)")

#### i: battery materials

In [ ]:
var = "inflow_battery_materials"

model = all_output["SSP2_M_CP"]
m = model.ev_battery[var].to_array().sum(["BatteryType", "Region", "Type"]).sel(time=slice(1972,None)).pint.to("kiloton") # convert to kg
m

In [ ]:
# Plot ev_battery: materials over time

df = df_ev_battery_mat_i.copy()
df = df.loc[(df["scenario"] != "Technology_Stagnation")] # otherwise the plot is too crowded

var = "inflow_battery_materials"

model = all_output["SSP2_M_CP"]
m = model.ev_battery[var].to_array().sum(["BatteryType", "Region", "Type"]).sel(time=slice(2000,2050)).pint.to("kiloton")

non_zero_materials = model.ev_battery["material_fractions"].material.where(
    (model.ev_battery["material_fractions"] != 0).any(dim=["Cohort", "BatteryType"]),
    drop=True).values

# color per material
colors = plt.cm.tab10.colors
color_map = {mat: colors[i % len(colors)] for i, mat in enumerate(non_zero_materials)}
# marker per (source, scenario)
markers = ["o", "X", "^", "D", "v", "P", "s", "*", "<", ">"]
marker_map = {}
marker_idx = 0

fig, ax = plt.subplots(figsize=(8,6))
for mat in list(non_zero_materials): # #["cobalt","manganese"]
    color = color_map[mat]
    plt.plot(m.time, m.sel(material=mat), label=str(mat), linewidth=2, color=color)
    if mat in df["material"].values:
        df_mat = df[df["material"] == mat]
        for (source, scenario), g in df_mat.groupby(["source", "scenario"]):
            if (source, scenario) not in marker_map:
                marker_map[(source, scenario)] = markers[marker_idx % len(markers)]
                marker_idx += 1
            plt.scatter(g.index, g["value"], color=color,  # same material color
                        edgecolors="black",  # black edge for better visibility
                        marker=marker_map[(source, scenario)],  # different marker
                        s=60, label=f"{source} – {scenario}", linewidth=1)
    
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.xlabel("Time", fontsize=14)
plt.ylabel(f"{var} in EV batteries (kt)", fontsize=14)
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
plt.title(f"{var} in EV batteries (SSP2_M_CP)", fontweight="bold", fontsize=14)
ax.grid(alpha=0.3)

#### energy density

In [ ]:
model = all_output["SSP2_M_CP"]
m = model.ev_battery["energy_density"].sum(["BatteryType"])
m
fig, ax = plt.subplots(figsize=(8,6))
plt.plot(m.Cohort, m, label="IMAGE-Materials - SSP2_M_CP", linewidth=1, c="black")

### Storage

#### s: Comp.: TIMER storage (GW)

In [ ]:
df = df_storage_power_s.copy()
df1 = df[(df["technology"] == "total")]
df1.reset_index(drop=False,inplace=True)

fig, ax = plt.subplots(figsize=(8,6))
ax.plot(df_power_timer.index, df_power_timer["global"], linewidth=2, c="black",
        label="TIMER_SSP2_M_CP - total storage")
ax.plot(df_power_timer_vlho.index, df_power_timer_vlho["global"], linewidth=2, c="black", linestyle="--",
        label="TIMER_SSP2_VLHO - total storage")
ax.plot(df_power_timer_seb.index, df_power_timer_seb["global"], linewidth=2, c="darkgray",
        label="TIMER SSP2_BL - total storage")
for row in range(len(df1)):
    src = df1["source"].iloc[row]
    scen = df1["scenario"].iloc[row]
    plt.scatter(df1["time"].iloc[row], df1["value"].iloc[row], label=f"{src} – {scen}", linewidth=1)

ax.set_xlabel("Time", fontsize=14)
ax.set_ylabel("Storage stocks (GW)", fontsize=14)
ax.tick_params(axis='both', labelsize=14)
ax.legend()
ax.grid(alpha=0.3)
ax.set_title("Global power capacity (stocks)", fontsize=14, fontweight="bold")
plt.show()

#### s: Comp.: TIMER storage (GWh)

In [ ]:
# TIMER GWh storage comparison
# comparison of SSP2_BL which Sebastiaan used to newer SSP2_M_CP runs + external data #

df = df_storage_energy_s.copy()
df1 = df[(df["source"] == "IRENA") & (df["category"] == "total")]
df1.reset_index(drop=False,inplace=True)

df2 = df[(df["category"] == "grid_scale_storage") & (df["technology"] == "battery")]
df2.reset_index(drop=False,inplace=True)

fig, ax = plt.subplots(figsize=(8,6))

ax.plot(df_energy_timer.index, df_energy_timer["global"], linewidth=2, c="black",
        label="TIMER_SSP2_M_CP - total storage")
ax.plot(df_energy_timer_vlho.index, df_energy_timer_vlho["global"], linewidth=2, c="black", linestyle="--",
        label="TIMER_SSP2_VLHO - total storage")
ax.plot(df_energy_timer_seb.index, df_energy_timer_seb["global"], linewidth=2, c="darkgray",
        label="TIMER SSP2_BL - total storage")
for row in range(len(df1)):
    src = df1["source"].iloc[row]
    scen = df1["scenario"].iloc[row]
    plt.scatter(df1["time"].iloc[row], df1["value"].iloc[row], label=f"{src} – {scen}", linewidth=1)
for row in range(len(df2)):
    src = df2["source"].iloc[row]
    scen = df2["scenario"].iloc[row]
    plt.scatter(df2["time"].iloc[row], df2["value"].iloc[row], label=f"{src} – {scen}", linewidth=1)

ax.tick_params(axis='both', labelsize=14)
ax.grid(alpha=0.3)

ax.set_xlabel("Time", fontsize=14)
ax.set_ylabel("Storage stocks (GWh)", fontsize=14)
ax.set_title("Global energy storage (stocks)", fontsize=14, fontweight="bold")
ax.legend()


#### s: Storage stock 3-tier

In [ ]:
# Plot storage (3-tier) - stacked #

model = all_output["SSP2_M_CP"]
unit = "GWh"
s_phs = model.elc_stor_phs["stocks"].sum(["Type", "Region"]).sel(Time=slice(2000,None)).pint.to(unit)
# check if PHS subtraction works correctly:
# s_0 = model.elc_stor_other["stocks_non_phs"].sum(["SuperType", "Region"]).pint.to(unit)
# s2 = s_0 + phs
s_other = model.elc_stor_other["stocks"].sum(["SuperType", "Region"]).sel(Time=slice(2000,None)).pint.to(unit)
s_v2g = model.ev_battery['stock_battery_kWh_v2g'].sum(["BatteryType","Region","Type"]).sel(Time=slice(2000,None)).pint.to(unit)
s_phs_v2g = s_v2g + s_phs
s_phs_v2g_other = s_v2g + s_phs + s_other

fig, ax = plt.subplots(figsize=(8,6))

time = s_phs.Time

ax.stackplot(
    time,
    s_phs,
    s_v2g,
    s_other,
    labels=[
        "PHS",
        "EV batteries (v2g)",
        "Dedicated storage"
    ]
)

# keep total reference line
plt.plot(
    df_energy_timer.index,
    df_energy_timer["global"],
    label="TIMER_SSP2_M_CP - total storage",
    color="black",
    # linestyle="--",
    linewidth=2.5
)

# ax.set_xlim(2000,2100)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.xlabel("Time", fontsize=14)
plt.ylabel(f"Storage stocks ({unit})", fontsize=14)
plt.title("Stocks of storage technologies", fontsize=14, fontweight="bold")
plt.legend()
plt.grid(alpha=0.3)

In [ ]:
# Plot storage (3-tier) #

model = all_output["SSP2_M_CP"]
unit = "GWh"
s_phs = model.elc_stor_phs["stocks"].sum(["Type", "Region"]).pint.to(unit)
# check if PHS subtraction works correctly:
# s_0 = model.elc_stor_other["stocks_non_phs"].sum(["SuperType", "Region"]).pint.to(unit)
# s2 = s_0 + phs
s_other = model.elc_stor_other["stocks"].sum(["SuperType", "Region"]).pint.to(unit)
s_v2g = model.ev_battery['stock_battery_kWh_v2g'].sum(["BatteryType","Region","Type"]).pint.to(unit)
s_phs_v2g = s_v2g + s_phs
s_phs_v2g_other = s_v2g + s_phs + s_other

fig, ax = plt.subplots(figsize=(8,6))
# plt.plot(s2.Time, s2, label="dedicated storage + ev batteries (v2g) + PHS", color="darkorange")
plt.plot(s_other.Time, s_other, label="dedicated storage", color="green")
plt.plot(s_v2g.Time, s_v2g, label="EV batteries (v2g)", color="darkorange")
plt.plot(s_phs.Time, s_phs, label="PHS", color="blue")
plt.plot(df_energy_timer.index, df_energy_timer["global"], label="TIMER_SSP2_M_CP - total storage", c="black", linestyle="--")
# plt.plot(test.Time, test, label="test", linestyle=":")
# plt.plot(df_energy_timer_seb.index, df_energy_timer_seb["global"], label="TIMER SSP2_BL - total storage", c="black", linestyle="--")

ax.set_xlim(2000,2100)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.xlabel("Time", fontsize=14)
plt.ylabel(f"Storage stocks ({unit})", fontsize=14)
plt.legend()
plt.grid(alpha=0.3)
plt.title(f"Stocks of storage technologies", fontsize=14, fontweight="bold")

In [ ]:
df = df_storage_energy_s.loc[df_storage_energy_s.index<2100]
df

#### s: PHS

In [ ]:
# Plot PHS storage #

df = df_storage_energy_s.loc[(df_storage_energy_s.index<2100) & (df_storage_energy_s["technology"] == "PHS")]

fig, ax = plt.subplots(figsize=(8,6))
# s_0 = model.elc_stor_other["stocks_0"].sum(["SuperType", "Region"]).pint.to("GWh")
# s = model.elc_stor_other["stocks"].sum(["SuperType", "Region"]).pint.to("GWh")
phs = model.elc_stor_phs["stocks"].sum(["Type", "Region"]).pint.to("GWh")

plt.plot(phs.Time, phs, label="IMAGE-Materials - SSP2_M_CP", c="black")
for (source, scenario), g in df.groupby(["source", "scenario"]):
    plt.scatter(g.index, g["value"], label=f"{source} – {scenario}", linewidth=1)

ax.set_xlim(2000,2100)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.xlabel("Year", fontsize=14)
plt.ylabel(f"PHS stocks (GWh)", fontsize=14)
plt.legend()
plt.grid(alpha=0.3)
plt.title(f"Stocks of Pumped Hydropower Storage", fontsize=14)

In [ ]:
model.ev_battery["stock_battery_kWh"].to_array()#.BatteryType.values

#### s: battery vs. vehicles materials

In [ ]:
# Compare materials stock: battery vs. vehicles #
import numpy as np
var = "stock_battery_materials"
m1 = model.vehicles["stock_by_cohort_materials"].sum(["Region", "Type"]).sel(Time=slice(1972,None))
m2 = model.ev_battery[var].to_array().sum(["BatteryType", "Region", "Type"]).sel(time=slice(1972,None))

common_materials = m1.material.values[np.isin(m1.material.values, m2.material.values)]
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

fig,ax = plt.subplots(figsize=(8,6))
for i, mat in enumerate(common_materials):
    color = colors[i % len(colors)]
    plt.plot(m1.Time, m1.sel(material=mat), label="m1 - " + str(mat), color=color)
    plt.plot(m2.time, m2.sel(material=mat), label="m2 - " + str(mat),color=color, linestyle=":", linewidth=2)
plt.xlabel("Year")
plt.ylabel(f"{var} in EV batteries (kg)")
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
plt.title(f"{var} in EV batteries")

In [ ]:
model = all_output["SSP2_M_CP"]
m1 = model.vehicles["stock_by_cohort"].sum(["Cohort", "Region"])
m1_bev = m1.sel(Type=m1.Type.str.contains("BEV")) #|PHEV
m1_phev = m1.sel(Type=m1.Type.str.contains("PHEV"))
cars_bev = m1_bev.sel(Type=m1_bev.Type.str.contains("Cars"))
cars_phev = m1_phev.sel(Type=m1_phev.Type.str.contains("Cars"))
buses_bev = m1_bev.sel(Type=m1_bev.Type.str.contains("Buses")).sum("Type")
buses_phev = m1_phev.sel(Type=m1_phev.Type.str.contains("Buses")).sum("Type")
trucks_bev = m1_bev.sel(Type=m1_bev.Type.str.contains("Trucks")).sum("Type")
trucks_phev = m1_phev.sel(Type=m1_phev.Type.str.contains("Trucks")).sum("Type")

In [ ]:
cars_bev

#### s: V2G

In [ ]:
# Plot ev_battery: input parameter over time
var = 'stock_battery_kWh_v2g'
unit = "GWh"
m = model.ev_battery[var].sum(["BatteryType","Region"]).sel(Time=slice(2000,None)).pint.to(unit)

fig, ax = plt.subplots(figsize=(8,6))
for t in m.Type.values: #["Cars - PHEV"]
    plt.plot(m.Time, m.sel(Type=t), label=str(t)) #

plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
ax.grid(alpha=0.3)

plt.xlabel("Time", fontsize=14)
plt.ylabel(f"V2G storage capacity ({unit})", fontsize=14)
plt.legend(fontsize=12)
plt.title("Energy capacity available for V2G storage per EV type", fontsize=14, fontweight="bold")

### vehicles

#### s: vehicles

In [ ]:
# Plot vehicles stock #

df = df_ev_vehicles_s.copy()

model = all_output["SSP2_M_CP"]
m1 = model.vehicles["stock_by_cohort"].sum(["Cohort", "Region"]).sel(Time=slice(2000,2060)) #2005,2031
m1_bev = m1.sel(Type=m1.Type.str.contains("BEV")) #|PHEV
m1_phev = m1.sel(Type=m1.Type.str.contains("PHEV"))
cars_bev = m1_bev.sel(Type=m1_bev.Type.str.contains("Cars")).sum("Type")
cars_phev = m1_phev.sel(Type=m1_phev.Type.str.contains("Cars")).sum("Type")
buses_bev = m1_bev.sel(Type=m1_bev.Type.str.contains("Buses")).sum("Type")
buses_phev = m1_phev.sel(Type=m1_phev.Type.str.contains("Buses")).sum("Type")
trucks_bev = m1_bev.sel(Type=m1_bev.Type.str.contains("Trucks")).sum("Type")
trucks_phev = m1_phev.sel(Type=m1_phev.Type.str.contains("Trucks")).sum("Type")

color_dict = {
    ("Cars", "BEV"): "#1f77b4",
    ("Cars", "PHEV"): "#aec7e8",
    ("Buses", "BEV"): "#2ca02c",
    ("Buses", "PHEV"): "#98df8a",
    ("Trucks", "BEV"): "#d62728",
    ("Trucks", "PHEV"): "#ff9896",
}


fig, ax = plt.subplots(figsize=(8,6))

plt.plot(cars_bev.Time, cars_bev, label="IMAGE-Materials - Cars - BEV", color=color_dict[("Cars", "BEV")])
plt.plot(cars_phev.Time, cars_phev, label="IMAGE-Materials - Cars - PHEV", color=color_dict[("Cars", "PHEV")])
plt.plot(buses_bev.Time, buses_bev, label="IMAGE-Materials - Buses - BEV", color=color_dict[("Buses", "BEV")])
plt.plot(buses_phev.Time, buses_phev, label="IMAGE-Materials - Buses - PHEV", color=color_dict[("Buses", "PHEV")])
plt.plot(trucks_bev.Time, trucks_bev, label="IMAGE-Materials - Trucks - BEV", color=color_dict[("Trucks", "BEV")])
plt.plot(trucks_phev.Time, trucks_phev, label="IMAGE-Materials - Trucks - PHEV", color=color_dict[("Trucks", "PHEV")])

for (source, scenario, tech, subtech), g in df.groupby(["source", "scenario", "Technology", "Sub-Technology"]):
    plt.scatter(g.index, g["value"], label=f"{source} – {scenario} – {tech} – {subtech}", color=color_dict[(tech, subtech)], linewidth=1)

# ax.set_ylim(0,200000000)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.xlabel("Year", fontsize=14)
plt.ylabel(f"EV vehicles stocks (count)", fontsize=14)
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
plt.grid(alpha=0.3)
plt.title(f"Stocks of EV Vehicles", fontsize=14)

In [ ]:
# Plot vehicles inflow #

df = df_ev_vehicles_i.copy()

model = all_output["SSP2_M_CP"]
m1 = model.vehicles["inflow"].to_array().sum(["Region"]).sel(time=slice(2005,2031)) #2005,2031
m1_bev = m1.sel(Type=m1.Type.str.contains("BEV")) #|PHEV
m1_phev = m1.sel(Type=m1.Type.str.contains("PHEV"))
cars_bev = m1_bev.sel(Type=m1_bev.Type.str.contains("Cars")).sum("Type")
cars_phev = m1_phev.sel(Type=m1_phev.Type.str.contains("Cars")).sum("Type")
buses_bev = m1_bev.sel(Type=m1_bev.Type.str.contains("Buses")).sum("Type")
buses_phev = m1_phev.sel(Type=m1_phev.Type.str.contains("Buses")).sum("Type")
trucks_bev = m1_bev.sel(Type=m1_bev.Type.str.contains("Trucks")).sum("Type")
trucks_phev = m1_phev.sel(Type=m1_phev.Type.str.contains("Trucks")).sum("Type")

color_dict = {
    ("Cars", "BEV"): "#1f77b4",
    ("Cars", "PHEV"): "#aec7e8",
    ("Buses", "BEV"): "#2ca02c",
    ("Buses", "PHEV"): "#98df8a",
    ("Trucks", "BEV"): "#d62728",
    ("Trucks", "PHEV"): "#ff9896",
}


fig, ax = plt.subplots(figsize=(8,6))

plt.plot(cars_bev.time, cars_bev, label="IMAGE-Materials - Cars - BEV", color=color_dict[("Cars", "BEV")])
plt.plot(cars_phev.time, cars_phev, label="IMAGE-Materials - Cars - PHEV", color=color_dict[("Cars", "PHEV")])
plt.plot(buses_bev.time, buses_bev, label="IMAGE-Materials - Buses - BEV", color=color_dict[("Buses", "BEV")])
plt.plot(buses_phev.time, buses_phev, label="IMAGE-Materials - Buses - PHEV", color=color_dict[("Buses", "PHEV")])
plt.plot(trucks_bev.time, trucks_bev, label="IMAGE-Materials - Trucks - BEV", color=color_dict[("Trucks", "BEV")])
plt.plot(trucks_phev.time, trucks_phev, label="IMAGE-Materials - Trucks - PHEV", color=color_dict[("Trucks", "PHEV")])

for (source, scenario, tech, subtech), g in df.groupby(["source", "scenario", "Technology", "Sub-Technology"]):
    plt.scatter(g.index, g["value"], label=f"{source} – {scenario} – {tech} – {subtech}", color=color_dict[(tech, subtech)], linewidth=1)

# ax.set_ylim(0,200000000)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.xlabel("Year", fontsize=14)
plt.ylabel(f"EV vehicles inflow (count)", fontsize=14)
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
plt.grid(alpha=0.3)
plt.title(f"Inflow of EV Vehicles", fontsize=14)

## old vs. new

#### i: battery materials

In [ ]:
# Plot ev_battery: materials over time
var = "inflow_battery_materials"

model = all_output_old["SSP2_M_CP"]
m1 = model.vehicles[var].sum(["battery", "Region", "Type"]).sel(Time=slice(1972,None))
model = all_output["SSP2_M_CP"]
m2 = model.ev_battery[var].to_array().sum(["BatteryType", "Region", "Type"]).sel(time=slice(1972,None))

for mat in m1.material.values[:4]: #
    plt.plot(m1.Time, m1.sel(material=mat), label="old - " + str(mat))
    plt.plot(m2.time, m2.sel(material=mat), label="new - " + str(mat), linestyle=":", linewidth=2)
    
plt.xlabel("Year")
plt.ylabel(f"{var} in EV batteries (kg)")
plt.legend()
plt.title(f"{var} in EV batteries")

In [ ]:
model.vehicles.keys()